# 02 · Lema, categoría gramatical y morfología

## Propósito

Ver las tres capas que agregan los componentes del pipeline, y entender por qué
la morfología es lo más rico que tiene el español y lo que la clase de text
mining no podía ver.

## Qué consume

- `datos/corpus.jsonl`

## Resultados de aprendizaje

- Explicar qué problema resuelve la lematización en un conteo de frecuencias.
- Saber por qué en español `token.tag_` no agrega nada y dónde está la
  información fina.
- Usar `token.morph` para filtrar por tiempo, persona o modo verbal.

## ◈ Microglosario

| Término | Qué es |
|---|---|
| **lema** | La forma de diccionario: infinitivo en los verbos, singular masculino en sustantivos y adjetivos |
| **POS** | *Part of speech*, la categoría gramatical: sustantivo, verbo, adjetivo. En spaCy, `token.pos_` |
| **`morph`** | Los rasgos finos de la palabra: tiempo, modo, persona, número, género |
| **rasgo morfológico** | Cada uno de esos datos por separado. `Tense=Past` es un rasgo |
| **forma** | La palabra tal como aparece en el texto, antes de lematizar. *Fuimos* es una forma; *ir* es su lema |

## Celda de reanudación

Si recién llegás o se te reinició el kernel, corré esta celda y seguís desde acá.

In [ ]:
# Recarga automática si editamos archivos externos (.py)
%load_ext autoreload
%autoreload 2

# spacy: la librería de análisis lingüístico.
import spacy

# pandas: para trabajar el corpus como una tabla.
import pandas as pd

# Counter: para contar cosas. Ya la usaste en la clase de text mining.
from collections import Counter

# es_core_news_lg: modelo en español (es), entrenado con noticias (news), grande (lg)
nlp = spacy.load("es_core_news_lg")
# lines=True: el archivo .jsonl tiene un registro JSON por renglón
corpus = pd.read_json("datos/corpus.jsonl", lines=True)

cantidad_parrafos = len(corpus)
# :, separa los miles con una coma
print("párrafos:", f"{cantidad_parrafos:,}")
# pipe_names: la lista de componentes que tiene el pipeline cargado
print("pipeline:", nlp.pipe_names)

if "lemmatizer" not in nlp.pipe_names:
    print("ATENCIÓN: sin lemmatizer no hay lemas, y este cuaderno entero es sobre lemas.")

## 1. Lematización

En la clase de text mining contaste palabras. El problema de contar palabras tal
como aparecen es que el español conjuga mucho: *es*, *era*, *fue*, *sea*, *siendo*
y *son* son seis palabras distintas para cualquier contador, y una sola para
cualquier lector.

El **lema** es la forma de diccionario: el infinitivo en los verbos, el singular
masculino en sustantivos y adjetivos.

In [ ]:
# .loc[1, "texto"]: el texto de la fila 1
parrafo = corpus.loc[1, "texto"]
doc = nlp(parrafo)

filas = []
for token in doc[:12]:  # doc[:12]: los primeros 12 tokens
    # lemma_: el lema, la forma de diccionario de la palabra
    fila = {"token": token.text, "lema": token.lemma_}
    filas.append(fila)

pd.DataFrame(filas)

### Cuántas formas junta un lema

Veámoslo sobre el corpus, no sobre un ejemplo. Vamos a recorrer 400 párrafos y
anotar, para cada lema, todas las formas distintas con las que aparece.

In [ ]:
primeros = corpus.head(400)
muestra = list(primeros["texto"])

# Un diccionario: para cada lema, el conjunto de formas en que apareció.
familias = {}

# Solo nos interesan estas categorías gramaticales.
categorias_que_nos_interesan = ("VERB", "AUX", "NOUN", "ADJ")

# nlp.pipe(): procesa muchos textos en lotes. disable=: apaga lo que no usamos acá.
componentes_apagados = ["parser", "ner"]

for documento in nlp.pipe(muestra, disable=componentes_apagados):
    for token in documento:

        # Primero verificamos que sea una palabra formada por letras.
        if token.is_alpha:

            # Después verificamos que nos interese su categoría gramatical.
            if token.pos_ in categorias_que_nos_interesan:
                lema = token.lemma_.lower()  # lower(): todo en minúsculas
                forma = token.text.lower()

                # set(): un conjunto. Como una lista, pero no guarda repetidos.
                if lema not in familias:
                    familias[lema] = set()

                familias[lema].add(forma)  # add(): agrega un elemento al conjunto

cantidad_de_lemas = len(familias)
print("lemas distintos encontrados:", cantidad_de_lemas)

Ya tenemos, para cada lema, el conjunto de formas con que apareció en el corpus.

Falta ordenarlos. El lema que reúna más formas distintas es el que más trabajo
hace: es donde la lematización más achica el vocabulario.

In [ ]:
# Armamos una lista de pares (cantidad, lema) para poder ordenarla.
para_ordenar = []
for lema in familias:
    cantidad = len(familias[lema])
    para_ordenar.append((cantidad, lema))

# sort() ordena los pares por su primer elemento: la cantidad.
# reverse=True ordena de mayor a menor.
para_ordenar.sort(reverse=True)

# [:5]: los primeros cinco
para_ordenar[:5]

Los números solos no dicen mucho. Lo que interesa mirar son **las formas concretas**
que el modelo juntó bajo un mismo lema.

In [ ]:
filas = []
for cantidad, lema in para_ordenar[:8]:  # [:8]: los ocho lemas con más formas
    formas_ordenadas = sorted(familias[lema])  # sorted(): en orden alfabético
    # join(): une las formas en un solo texto, separadas por ", "
    formas_como_texto = ", ".join(formas_ordenadas)
    fila = {
        "lema": lema,
        "formas": cantidad,
        "cuales": formas_como_texto,
    }
    filas.append(fila)

pd.DataFrame(filas)

Sin lematizar, cada una de esas formas cuenta por separado. Con lemas, se
juntan. Midamos cuánto achica eso el vocabulario.

In [ ]:
formas_distintas = set()
lemas_distintos = set()

componentes_apagados = ["parser", "ner"]

for documento in nlp.pipe(muestra, disable=componentes_apagados):
    for token in documento:

        # Solo nos interesan los tokens que son palabras (no signos de puntuación, ni números sueltos).
        if token.is_alpha:
            forma = token.text.lower()
            lema = token.lemma_.lower()

            formas_distintas.add(forma)
            lemas_distintos.add(lema)

cantidad_formas = len(formas_distintas)
cantidad_lemas = len(lemas_distintos)
reduccion = 100 * (1 - cantidad_lemas / cantidad_formas)

print("formas distintas :", f"{cantidad_formas:,}")
print("lemas distintos  :", f"{cantidad_lemas:,}")
# :.0f: sin decimales
print("reducción        :", f"{reduccion:.0f}%")

## 2. Categoría gramatical (POS)

`token.pos_` dice qué es cada palabra: sustantivo, verbo, adjetivo, determinante.
Las etiquetas son las de [Universal Dependencies](https://universaldependencies.org/u/pos/),
las mismas para todos los idiomas.

In [ ]:
filas = []
for token in doc[:12]:
    # pos_: la categoría gramatical. spacy.explain(): la traduce a una descripción
    explicacion = spacy.explain(token.pos_)
    fila = {
        "token": token.text,
        "pos": token.pos_,
        "que_significa": explicacion,
    }
    filas.append(fila)

pd.DataFrame(filas)

`spacy.explain()` responde en inglés. Es una limitación de la librería, no
tuya: las descripciones vienen así. Para la cursada podés tener a mano una tabla
de traducción de las etiquetas que más uses.

### El detalle que confunde: `tag_` no sirve en español

spaCy tiene dos atributos de categoría gramatical: `pos_` (universal) y `tag_`
(específico del idioma). En inglés son distintos y `tag_` es más fino: `NOUN`
contra `NN`, `NNS`, `NNP`.

En español, no.

In [ ]:
# tag_: la etiqueta detallada que da el modelo, además de pos_
iguales = 0
for token in doc:
    if token.pos_ == token.tag_:
        iguales = iguales + 1

cantidad_de_tokens = len(doc)
print("tokens donde pos_ == tag_:", iguales, "de", cantidad_de_tokens)

filas = []
for token in doc[:8]:
    fila = {"token": token.text, "pos": token.pos_, "tag": token.tag_}
    filas.append(fila)

pd.DataFrame(filas)

**Idénticos.** Los modelos españoles de spaCy usan un componente llamado
`morphologizer` en lugar de un `tagger` con etiquetas propias, así que `tag_` es
una copia de `pos_`.

Si alguna vez ves un tutorial que arma una tabla "POS simplificado / POS
detallado" con estas dos columnas, en español esa tabla no dice nada.

La información fina existe, pero está en otro lado.

## 3. Morfología

`token.morph` trae los rasgos morfológicos: género, número, persona, tiempo,
modo. Es la capa más rica del español y la que ningún conteo de palabras puede
ver.

In [ ]:
# Solo nos interesan estas categorías gramaticales.
categorias_que_nos_interesan = ("VERB", "AUX", "NOUN", "DET", "ADJ", "PRON")

filas = []
for token in doc[:14]:
    if token.pos_ in categorias_que_nos_interesan:
        # morph: los rasgos morfológicos. str() los convierte en texto para la tabla
        morfologia_como_texto = str(token.morph)
        fila = {
            "token": token.text,
            "pos": token.pos_,
            "morfologia": morfologia_como_texto,
        }
        filas.append(fila)

pd.DataFrame(filas)

Cada rasgo se puede consultar por separado con `.get()`, que devuelve una
lista (vacía si el rasgo no aplica).

In [ ]:
ejemplo = nlp("Ayer preparábamos milanesas, hoy preparo fideos y mañana comeremos afuera.")

filas = []
for token in ejemplo:

    # Solo nos interesan los verbos (VERB) y los auxiliares (AUX).
    if token.pos_ in ("VERB", "AUX"):
        # morph.get("Tense"): el rasgo tiempo, como lista (vacía si no lo tiene)
        tiempo = token.morph.get("Tense")
        persona = token.morph.get("Person")
        modo = token.morph.get("Mood")
        fila = {
            "verbo": token.text,
            "tiempo": tiempo,
            "persona": persona,
            "modo": modo,
        }
        filas.append(fila)

pd.DataFrame(filas)

## 4. Atributos del token

Además de lema, POS y morfología, cada token trae banderas útiles para filtrar.

In [ ]:
filas = []
for token in doc[:10]:
    fila = {
        "token": token.text,
        "is_alpha": token.is_alpha,  # ¿está formado solo por letras?
        "is_stop": token.is_stop,  # ¿es una palabra vacía (stopword)?
        "is_punct": token.is_punct,  # ¿es un signo de puntuación?
        "like_num": token.like_num,  # ¿parece un número ("3", "tres")?
    }
    filas.append(fila)

pd.DataFrame(filas)

`is_stop` marca las palabras vacías. En text mining vos armaste esa lista a
mano, mirando el top de frecuencias y decidiendo qué sacar. spaCy trae una lista
adentro.

**Pero es una lista que alguien escribió**, y conviene abrirla antes de confiar.

In [ ]:
# Defaults.stop_words: el conjunto de palabras vacías que trae el modelo
palabras_vacias = nlp.Defaults.stop_words
cantidad_de_palabras_vacias = len(palabras_vacias)

print("palabras vacías en el modelo español:", cantidad_de_palabras_vacias)
print()

pronombres_a_revisar = ["tú", "usted", "ustedes", "vosotros", "él", "yo", "vos"]
for palabra in pronombres_a_revisar:
    esta = palabra in palabras_vacias
    print(" ", palabra, "->", esta)

Mirá bien esa lista. **Están todos los pronombres de segunda persona del
español menos uno: `vos`.** También están `tú`, `usted`, `ustedes` y `vosotros`.

No es un olvido caprichoso: la lista se armó pensando en el español peninsular.
La consecuencia práctica es concreta — si filtrás palabras vacías en un corpus
rioplatense, *tú* desaparece de tus conteos y *vos* se queda, y vas a terminar
comparando peras con manzanas.

Se puede corregir, tocando el vocabulario del modelo.

In [ ]:
# nlp.vocab["palabra"] devuelve el lexema: la entrada de esa palabra
# en el vocabulario del modelo. Ahí vive el atributo is_stop.
lexema_vos = nlp.vocab["vos"]
lexema_vos.is_stop = True

# Ojo: el lexema de "Vos" con mayúscula es OTRO. Hay que marcarlo aparte.
lexema_vos_mayuscula = nlp.vocab["Vos"]
lexema_vos_mayuscula.is_stop = True

prueba = nlp("Vos sabés que vos siempre tenés razón.")

for token in prueba:
    print(" ", token.text, "-> is_stop =", token.is_stop)

> **Por qué se hace así y no con `nlp.Defaults.stop_words.add()`.**
> Ese otro camino aparece en muchos tutoriales y funciona **solo a veces**: si el
> modelo ya vio esa palabra, su lexema quedó guardado con el valor viejo y el
> cambio no tiene efecto. O sea que el mismo código anda o no anda según qué
> celdas corriste antes. Tocar `nlp.vocab` funciona siempre.

**Ojo con lo que acabás de hacer.** Sacar *vos* de tus conteos es una decisión
de análisis, no de limpieza. Si estudiás cómo Casciari construye la voz narrativa,
*vos* es probablemente el dato más importante del corpus y acabás de borrarlo.

Cualquier cosa que agregues o saques de esa lista tiene que estar anotada y
justificada en tu bitácora.

In [ ]:
# La deshacemos, para seguir el cuaderno con el modelo tal como viene.
lexema_vos = nlp.vocab["vos"]
lexema_vos.is_stop = False

lexema_vos_mayuscula = nlp.vocab["Vos"]
lexema_vos_mayuscula.is_stop = False

prueba = nlp("Vos sabés que vos siempre tenés razón.")
for token in prueba[:4]:  # [:4]: los primeros cuatro tokens
    print(" ", token.text, "-> is_stop =", token.is_stop)

## 5. Aplicación: ¿cómo narra Casciari, y cambia con los años?

Esta es una pregunta de análisis literario, y `morph` la puede contestar en
números. Vamos a mirar la distribución de tiempos verbales por año.

In [ ]:
def tiempos_de_un_anio(anio, cuantos_parrafos):
    """Devuelve el porcentaje de cada tiempo verbal en una muestra de un año."""
    # Para cada fila, ¿es de este año? Da True o False. Nos quedamos con las True.
    es_del_anio = corpus["anio"] == anio
    seleccion = corpus[es_del_anio]
    primeros = seleccion.head(cuantos_parrafos)
    textos = list(primeros["texto"])

    componentes_apagados = ["parser", "ner"]

    conteo = Counter()
    for documento in nlp.pipe(textos, disable=componentes_apagados):
        for token in documento:

            # Solo nos interesan los verbos.
            if token.pos_ == "VERB":
                tiempos = token.morph.get("Tense")

                # Y solo los que tienen un tiempo verbal marcado.
                if tiempos:
                    tiempo = tiempos[0]  # morph.get() devuelve una lista: tomamos el primero
                    conteo[tiempo] = conteo[tiempo] + 1

    # values(): las cantidades del Counter. sum(): las suma
    cantidades = conteo.values()
    total = sum(cantidades)

    fila = {"anio": anio, "verbos": total}
    for tiempo in conteo:
        porcentaje = 100 * conteo[tiempo] / total
        fila[tiempo] = round(porcentaje, 1)  # round(..., 1): redondea a un decimal
    return fila

La función trabaja sobre **un** año. Ahora la corremos sobre cuatro, espaciados a lo
largo del corpus, con la misma cantidad de párrafos en cada uno.

Que la muestra sea del mismo tamaño no es un detalle: si no, estaríamos comparando
cuánto escribió cada año en vez de cómo escribió.

In [ ]:
anios_elegidos = [2004, 2008, 2011, 2015]

filas = []
for anio in anios_elegidos:
    fila = tiempos_de_un_anio(anio=anio, cuantos_parrafos=110)
    filas.append(fila)

pd.DataFrame(filas)

`Pres` es presente, `Past` pretérito perfecto simple, `Imp` imperfecto y `Fut`
futuro.

Antes de interpretar, dos advertencias:

1. Son **110 párrafos por año**, no el año completo. Es una muestra, y una
   muestra chica.
2. Los párrafos están tomados con `.head()`, o sea los primeros del archivo. No
   es una muestra aleatoria.

Las dos cosas se pueden arreglar y el ejercicio te pide que lo hagas.

## 6. Ejercicio

1. Modificá `tiempos_de_un_anio` para que tome una muestra **aleatoria** en lugar
   de los primeros párrafos. Pista: `seleccion.sample(cuantos_parrafos, random_state=42)`.
   ¿Cambian mucho los porcentajes?
2. Corré la tabla para **los doce años** y mirá si hay alguna tendencia.
3. Hacé lo mismo con la **persona** gramatical (`Person`) en lugar del tiempo.
   ¿En qué persona narra Casciari? ¿Cambia?

Escribí abajo qué encontraste, en dos o tres oraciones. Incluí lo que **no**
podés concluir con esta muestra.

In [ ]:
# TU CÓDIGO ACÁ
#
# Paso 1: copiá acá la función tiempos_de_un_anio y ponele otro nombre,
#         por ejemplo tiempos_al_azar.
#
# Paso 2: adentro de la función, cambiá la línea
#             primeros = seleccion.head(cuantos_parrafos)
#         por
#             primeros = seleccion.sample(n=cuantos_parrafos, random_state=42)
#         sample(): elige filas al azar. random_state=42: el azar es siempre el
#         mismo, así el resultado se puede repetir.
#
# Paso 3: armá la tabla con los doce años, como en la sección 5, pero con
#             anios_elegidos = [2004, 2005, 2006, ..., 2015]
#
# Paso 4: para la persona gramatical, hacé una copia más de la función y
#         cambiá "Tense" por "Person".

> ### Perilla
>
> El modelo que cargamos es `es_core_news_lg`, de 631 MB. Existen dos hermanos
> más chicos: `es_core_news_md` (54 MB) y `es_core_news_sm` (16 MB).
>
> La celda de abajo corre la misma frase rioplatense con el modelo grande.
> **Antes de correrla, anotá qué esperás que pase si la corrieras con `sm`.**
>
> En el cuaderno `04` vamos a correr los tres y comparar. La diferencia entre lo
> que esperabas y lo que pasa es el contenido de ese cuaderno.

In [ ]:
frase = "Vos sabés que tenés razón, che. Mirá, vení y contame qué te parece."
doc_voseo = nlp(frase)

filas = []
for token in doc_voseo:

    # Descartamos los signos de puntuación (is_punct).
    if not token.is_punct:
        fila = {"token": token.text, "pos": token.pos_, "lema": token.lemma_}
        filas.append(fila)

pd.DataFrame(filas)

## Lo que hiciste hoy

- Viste cuántas formas junta un lema y cuánto achica el vocabulario.
- Aprendiste que en español `tag_` es una copia de `pos_` y que la información
  fina está en `morph`.
- Usaste `morph` para contestar una pregunta que un conteo de palabras no puede.
- Abriste la lista de palabras vacías del modelo y viste que tiene supuestos
  sobre qué español se habla.

## Lo que viene

En el `03` pasamos de la palabra a la oración: relaciones sintácticas, grupos
nominales y entidades nombradas.